# Aegis - Phase 9: Agent Indirect-Injection (under obfuscation)

Two CPU-only agent-security evaluations:

1. **Scenario benchmark** - injected instructions hidden in untrusted content (email / web / calendar / document / tool-poisoning) plus benign controls. Reports injection-detection, dangerous-action-blocked, and benign-pass rates.
2. **Injection under obfuscation** - the agent-side of Aegis's niche. The same injections are hidden with Base64 / homoglyph / zero-width / character-spacing, and we compare detection by Aegis (which de-obfuscates at L0 *before* the injection rules) against a regex-only scanner that does not. This is where L0 normalization - including the character-spacing fix - earns its keep against disguised indirect injections.

**Setup:** CPU is fine, no secrets or datasets needed. Then Run All.

(The field-standard AgentDojo benchmark is provided as `eval.agent_eval.run_agentdojo`, which needs the `agentdojo` package plus an LLM backend; run it when you have API access.)

In [ ]:
import sys, os, glob, subprocess
REPO_URL = "https://github.com/g25ait2149/aegis.git"     # same repo as the other notebooks
DEST = "/kaggle/working/aegis_src"
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DEST], check=False)
hits = glob.glob(DEST + "/**/aegis/__init__.py", recursive=True)
root = os.path.dirname(os.path.dirname(hits[0])) if hits else DEST
sys.path.insert(0, root)
for m in [m for m in sys.modules if m == "aegis" or m.startswith(("aegis.", "eval"))]:
    del sys.modules[m]
print("aegis repo at:", root)

In [ ]:
from eval.agent_eval import run_agent_eval, run_agent_robustness

print("=== Scenario benchmark (email / web / calendar / doc / tool-poison) ===")
run_agent_eval()

print("\n=== Indirect injection under obfuscation ===")
rows, benign = run_agent_robustness()

## What to read
- **Scenario benchmark**: detection / dangerous-action-block near 1.0 with benign-pass near 1.0.
- **Under obfuscation**: `detect_aegis` should stay high across all channels while `detect_regex_only` collapses to ~0 on Base64 / zero-width / character-spacing - because a regex-only scanner never sees past the disguise, while Aegis normalizes first. Benign content is not flagged.

## Next
- Field-standard agent benchmark: `run_agentdojo()` (needs `pip install agentdojo` + an LLM backend).